# DuckPD MOMENT Time-Series Embeddings Walkthrough

This interactive notebook loads the pinned [AutonLab/MOMENT-1-small](https://huggingface.co/AutonLab/MOMENT-1-small) checkpoint through DuckPD's built-in MOMENT backend. It reads a bounded OHLCV slice from DuckPD's published demo feature store, builds ordered rolling return windows, runs bounded GPU inference, persists model identity with the vectors, and performs exact time-series retrieval.

### Highlights
- **Existing demo data**: Read minute OHLCV features from the published `hifinab/fdb` feature store.
- **Built-in model adapter**: Use `MomentEmbeddingProvider` through the same model registration and preparation lifecycle as text embeddings.
- **Immutable model identity**: Pin the Hugging Face checkpoint revision and declare the input and pooling contract.
- **Explicit accelerator selection**: Request `device="cuda"`; the provider raises instead of silently falling back to CPU.
- **Bounded in-engine inference**: DuckPD sends complete `float32[512]` Arrow windows to MOMENT in bounded batches.
- **Persisted representation identity**: Reload Parquet vectors and search without repeating the representation contract.

> MOMENT is demonstrated as an integration example, not a DuckPD-endorsed retrieval model. Qualify learned representations against a relevant native baseline before production use.


## 1. Select the accelerator kernel and configure the model

In VS Code, choose **Select Kernel** in the upper-right and select **DuckPD ROCm 7.2.4**. The setup command in `demo/generate_data/README.md` installs the pinned MOMENT runtime into that environment. NVIDIA users can use an equivalent CUDA-enabled PyTorch environment. Export `HF_TOKEN` into the kernel environment when the published demo dataset requires authentication; the notebook passes it only to `FeatureStore`.

MOMENT-1-small accepts 512 observations and produces a 512-dimensional sequence embedding. The `EmbeddingModelSpec` records checkpoint, mean pooling, input role, internal RevIN normalization, and final unit-normalization semantics without importing or loading the model during planning.


In [1]:
import importlib
import os
import sys
from pathlib import Path
from time import perf_counter

import duckpd as pd

try:
    torch = importlib.import_module("torch")
    importlib.import_module("huggingface_hub")
    importlib.import_module("momentfm")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "This notebook is running the wrong kernel or lacks the MOMENT runtime. "
        f"Current interpreter: {sys.executable}. Follow the MOMENT setup in "
        "demo/generate_data/README.md, then restart the kernel."
    ) from error
if not torch.cuda.is_available():
    raise RuntimeError(
        f"PyTorch in {sys.executable} cannot access a GPU; DuckPD will not fall back to CPU."
    )

MODEL_ID = "AutonLab/MOMENT-1-small"
MODEL_REVISION = "411e288267f82cce86296dbe4d6c8bc533cc162f"
FEATURE_STORE_SOURCE = "hf://datasets/hifinab/fdb"
DATA_START = "2024-01-02T08:00:00Z"
DATA_END = "2024-01-03T08:35:00Z"
TICKERS = ("001", "002", "003")
WINDOW = 512
DIMENSION = 512
DEVICE = "cuda"
PROVIDER_BATCH_SIZE = 32

DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path("..")
ARTIFACT_DIR = DEMO_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
FEATURE_CACHE = DEMO_DIR / ".cache" / "fdb"
EMBEDDED_DATA = ARTIFACT_DIR / "moment-provider-small-ohlcv-embedded.parquet"
EMBEDDED_SIDECAR = Path(f"{EMBEDDED_DATA}.duckpd-embeddings.json")

MODEL = pd.embedding_model(
    MODEL_ID,
    revision=MODEL_REVISION,
    dimension=DIMENSION,
    backend="moment",
    normalize=True,
    pooling="mean",
    input=pd.series_embedding_input(
        length=WINDOW,
        channels=("bar_return",),
        roles=("target",),
        normalization="moment-revin-affine-false-v1",
    ),
)
REPRESENTATION = pd.series_representation(
    window=WINDOW,
    channels=("bar_return",),
    sampling="observations",
    data_contract="hifinab/fdb/ohlcv/bar-return/v1",
    normalization="none",
    unit_norm=True,
    zero_scale="error",
    encoder=MODEL,
)

print(f"Kernel interpreter: {sys.executable}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"DuckPD version: {pd.__version__}")
print(f"Model revision: {MODEL.revision}")
print(f"Representation dimension: {REPRESENTATION.dimension}")
print(f"Representation fingerprint: {REPRESENTATION.fingerprint}")

Kernel interpreter: /home/hi/duckpd/demo/generate_data/.venv-rocm/bin/python
GPU: AMD Radeon Graphics
DuckPD version: 0.1.4
Model revision: 411e288267f82cce86296dbe4d6c8bc533cc162f
Representation dimension: 512
Representation fingerprint: aa8960788a1ab80aa99bbda300c952afe22c1679bd0100c3848f7c6bb3489be4


## 2. Prepare the model on the GPU

`MomentEmbeddingProvider` is DuckPD's built-in adapter for MOMENT checkpoints. Registration selects the explicit GPU runtime without loading the model. Preparation then downloads or verifies the immutable checkpoint cache, initializes embedding mode, and reports the actual PyTorch execution provider.


In [2]:
session = pd.connect(memory_limit="1GB", threads=4)
provider = pd.MomentEmbeddingProvider(MODEL, device=DEVICE)
session.register_embedding_provider(MODEL, provider)

preparation_started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - preparation_started

print(f"Backend: {prepared.backend} via {prepared.execution_providers}")
print(f"Model cache: {prepared.cache_path}")
print(f"Model preparation time: {preparation_seconds:.3f}s")
print(f"Prepared model fingerprint: {prepared.model_fingerprint}")

Loading weights from local directory
Backend: moment via ('PyTorchROCm',)
Model cache: /home/hi/.cache/duckpd/embeddings/7b03899014f1c16015ee9b3f07de0ef2f2a643d5927637a3a0c2d334571bf86d
Model preparation time: 0.644s
Prepared model fingerprint: 7b03899014f1c16015ee9b3f07de0ef2f2a643d5927637a3a0c2d334571bf86d


## 3. Read demo OHLCV data and build rolling return windows

The source is the same published `hifinab/fdb` feature store used by the main FeatureStore walkthrough. The bounded slice contains 545 minute bars for each of three tickers: the full 2 January 2024 session plus the first 35 minutes of the next session. Each ticker therefore produces 34 complete 512-observation windows. Grouped rolling prevents history from crossing ticker boundaries; warm-up rows remain null and never reach the provider.


In [3]:
store = pd.FeatureStore(
    source=FEATURE_STORE_SOURCE,
    cache=FEATURE_CACHE,
    token=os.getenv("HF_TOKEN"),
    session=session,
)
prices = store.features(
    features={"open": "ohlcv:open", "close": "ohlcv:close"},
    start=DATA_START,
    end=DATA_END,
    filters={"ticker": list(TICKERS)},
    alignment="exact",
    order_by=["ticker", "datetime"],
)
returns = prices.assign(
    bar_return=lambda frame: ((frame["close"] - frame["open"]) / frame["open"]).astype("float32")
)
windows = returns.assign(
    return_window=lambda frame: frame.groupby("ticker")["bar_return"].rolling(WINDOW).to_array()
)

bars_per_ticker = 545
print(f"Demo source: {FEATURE_STORE_SOURCE}")
print(f"Requested rows: {len(TICKERS) * bars_per_ticker:,}")
print(f"Complete windows: {len(TICKERS) * (bars_per_ticker - WINDOW + 1):,}")
print(f"Executions after window planning: {session.execution_count}")

Demo source: hf://datasets/hifinab/fdb
Requested rows: 1,635
Complete windows: 102
Executions after window planning: 1


## 4. Load or build the MOMENT-embedded dataset

If the Parquet file and DuckPD sidecar already exist, this step restores them without recomputing vectors. Otherwise `embed_series()` adds a lazy learned representation and `write_parquet()` streams complete windows through MOMENT in bounded batches. The sidecar binds `moment_embedding` to the full representation fingerprint.


In [4]:
if EMBEDDED_DATA.exists() and EMBEDDED_SIDECAR.exists():
    dataset_status = f"Loaded {EMBEDDED_DATA}"
else:
    build_started = perf_counter()
    embedded_plan = windows.embed_series(
        columns={"bar_return": "return_window"},
        into="moment_embedding",
        representation=REPRESENTATION,
        batch_size=PROVIDER_BATCH_SIZE,
    )
    embedded_plan.write_parquet(EMBEDDED_DATA, overwrite=True)
    build_seconds = perf_counter() - build_started
    dataset_status = f"Created {EMBEDDED_DATA} in {build_seconds:.3f} seconds"

embedded = session.read_parquet(EMBEDDED_DATA)
print(dataset_status)
print(f"Columns: {embedded.columns}")

Created ../.artifacts/moment-provider-small-ohlcv-embedded.parquet in 0.881 seconds
Columns: ('datetime', 'ticker', 'open', 'close', 'bar_return', 'return_window', 'moment_embedding')


## 5. Preview the source columns

The 512-return raw windows and 512-dimensional embeddings remain in the lazy frame. This bounded preview omits both large columns and shows the null warm-up boundary for ticker `001`.


In [5]:
preview = embedded[embedded["ticker"] == "001"][
    ["ticker", "datetime", "open", "close", "bar_return"]
].head(WINDOW + 2)
display(preview.tail(4))

,ticker,datetime,open,close,bar_return
510,001,2024-01-03 08:00:00+00:00,356.975467,356.668403,-0.000860
511,001,2024-01-03 08:01:00+00:00,356.681725,356.268458,-0.001159
512,001,2024-01-03 08:02:00+00:00,356.320459,357.043497,0.002029
513,001,2024-01-03 08:03:00+00:00,356.932823,356.570143,-0.001016


## 6. Search with a raw time-series query

Use ticker `001`'s first complete return window as a query-by-example. `search_series()` reads the representation identity restored from the sidecar, applies the same MOMENT provider to the raw query, and ranks ticker `001`'s complete demo-data windows by exact cosine distance. Passing `REPRESENTATION` in the second plan is a fingerprint compatibility assertion, not an override.


In [6]:
candidates = embedded[(embedded["moment_embedding"].notna()) & (embedded["ticker"] == "001")]
query_row = candidates[["ticker", "datetime", "return_window"]].head(1)
query = {"bar_return": tuple(float(value) for value in query_row.iloc[0]["return_window"])}

query_started = perf_counter()
matches = candidates.vector.search_series(
    query,
    column="moment_embedding",
    metric="cosine",
    k=8,
    tie_breaker="datetime",
)[["ticker", "datetime", "open", "close", "bar_return", "_distance"]]
explicit_matches = candidates.vector.search_series(
    query,
    column="moment_embedding",
    representation=REPRESENTATION,
    metric="cosine",
    k=8,
    tie_breaker="datetime",
)[["ticker", "datetime", "open", "close", "bar_return", "_distance"]]

result = matches.collect()
explicit_result = explicit_matches.collect()
assert result.equals(explicit_result)
assert result.iloc[0]["ticker"] == query_row.iloc[0]["ticker"]
assert result.iloc[0]["datetime"] == query_row.iloc[0]["datetime"]
assert abs(float(result.iloc[0]["_distance"])) < 1e-5
query_seconds = perf_counter() - query_started

print(f"Query ticker: {query_row.iloc[0]['ticker']}")
print(f"Query endpoint: {query_row.iloc[0]['datetime']}")
print(f"Query-to-response for inferred and explicit plans: {query_seconds:.3f} seconds")
print("Sidecar-inferred and explicit-representation results are identical.")

Query ticker: 001
Query endpoint: 2024-01-03 08:01:00+00:00
Query-to-response for inferred and explicit plans: 0.066 seconds
Sidecar-inferred and explicit-representation results are identical.


## 7. Inspect the results

Lower cosine distance means a nearer MOMENT representation. The query window itself appears first at distance zero. Interpret the remaining ordering only as this pinned adapter's geometry over the demo OHLCV returns; a production adoption still needs held-out retrieval evaluation against a native representation.


In [7]:
display(result)
session.close()
print(f"Reusable vectors: {EMBEDDED_DATA}")
print(f"Representation sidecar: {EMBEDDED_SIDECAR}")

,ticker,datetime,open,close,bar_return,_distance
0,001,2024-01-03 08:01:00+00:00,356.681725,356.268458,-0.001159,0.000000
1,001,2024-01-03 08:17:00+00:00,356.044395,356.299356,0.000716,0.000095
2,001,2024-01-03 08:09:00+00:00,354.546448,355.214447,0.001884,0.000110
3,001,2024-01-03 08:25:00+00:00,355.662407,356.220671,0.001570,0.000151
4,001,2024-01-03 08:33:00+00:00,360.516687,360.901053,0.001066,0.000499
5,001,2024-01-03 08:16:00+00:00,355.225538,356.144036,0.002586,0.002416
6,001,2024-01-03 08:24:00+00:00,355.659698,355.762001,0.000288,0.002571
7,001,2024-01-03 08:08:00+00:00,355.425001,354.507648,-0.002581,0.002677


Reusable vectors: ../.artifacts/moment-provider-small-ohlcv-embedded.parquet
Representation sidecar: ../.artifacts/moment-provider-small-ohlcv-embedded.parquet.duckpd-embeddings.json
